# CUDA Graph的原理演示

通过PyTorch提供的CUDA API来创建图编译与运算，通过了解其基本操作流程来理解其在推理中应用场景。



Author: kaiyuan

Email: kyxie@zju.edu.cn

镜像拉取：
```
docker pull nvcr.io/nvidia/sglang:26.01-py3
```
版本介绍：[Link](https://catalog.ngc.nvidia.com/orgs/nvidia/containers/sglang?version=26.01-py3)

容器创建示例：
```
docker run -itd --rm --gpus all --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 \
-v /data/nfs/kaiyuan:/data/nfs/kaiyuan \
--name sglang-dev nvcr.io/nvidia/sglang:26.01-py3 bash
```

本例测试机器信息：
- NVIDIA A100-SXM4-80GB
- NVIDIA-SMI 570.172.08
- Driver Version: 570.172.08
- CUDA Version: 13.1


## 1 图捕获与重放

构建一个简单model，完成图捕获与重放。

- 图捕获：每个 batch size 对应一个独立的 CUDAGraph，因为图固定了张量的形状和内存地址。
- 捕获时使用固定的 static_input和static_output，它们的存储地址被记录在图内；

- 重放前通过 .copy_() 将新数据写入static_input，保证内存地址不变；

- 重放后结果直接出现在static_output中。

In [ ]:
import torch
import torch.nn as nn
import random

# 检查 CUDA 是否可用
assert torch.cuda.is_available(), "CUDA不可用!本示例需要GPU"
device = torch.device('cuda')

# -------------------- 定义模型 --------------------
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(32, 64)   # 输入特征 32 -> 64
        self.fc2 = nn.Linear(64, 32)   # 64 -> 32
        self.relu = nn.ReLU()

    def forward(self, x):
        # x 形状: [bs, 8, 32]
        x = self.fc1(x)                # [bs, 8, 64]
        x = self.relu(x)
        x = self.fc2(x)                # [bs, 8, 32]
        return x

model = SimpleModel().to(device)
model.eval()  # 推理模式，关闭dropout/batchnorm等随机行为

# -------------------- 为不同batch size捕获图 --------------------
batch_sizes = [1, 2, 4, 8, 16]
graph_pool = {}  # 字典：bs -> (graph, static_input, static_output)

# 预热 CUDA 上下文，避免捕获时包含自动调优开销
warmup_input = torch.randn(8, 8, 32, device=device)
for _ in range(3):
    _ = model(warmup_input)
torch.cuda.synchronize()

for bs in batch_sizes:
    # 创建固定的输入、输出占位张量（图会记住它们的地址）
    static_input = torch.randn(bs, 8, 32, device=device)
    static_output = torch.empty_like(static_input)  # 形状与输入相同

    # 开始捕获
    graph = torch.cuda.CUDAGraph()
    with torch.cuda.graph(graph):
        # 在此上下文中执行的所有 CUDA 操作都会被捕获
        static_output = model(static_input)

    # 将图及相关张量保存到池中
    graph_pool[bs] = (graph, static_input, static_output)

print(f"已为 batch sizes {batch_sizes} 捕获图完成。")

# -------------------- 模拟多次推理，随机选择batch size --------------------
num_iterations = 10
for i in range(num_iterations):
    # 随机选择一个batch size
    bs = random.choice(batch_sizes)
    graph, static_input, static_output = graph_pool[bs]

    # 生成新的随机输入数据
    new_input = torch.randn(bs, 8, 32, device=device)

    # 将新数据复制到图使用的静态输入张量中（in-place 操作，不改变地址）
    static_input.copy_(new_input)

    # 重放图
    graph.replay()

    # 此时 static_output 已经更新为对应新输入的计算结果
    # 可以取出结果用于后续处理，例如与普通前向结果对比验证
    with torch.no_grad():
        expected_output = model(new_input)

    # 验证结果是否一致（允许微小误差）
    if torch.allclose(static_output, expected_output, atol=1e-5):
        print(f"迭代 {i+1}: bs={bs} 结果与一致")
    else:
        print(f"迭代 {i+1}: bs={bs} 结果不一致！")


已为 batch sizes [1, 2, 4, 8, 16] 捕获图完成。
迭代 1: bs=4 结果与一致
迭代 2: bs=2 结果与一致
迭代 3: bs=4 结果与一致
迭代 4: bs=1 结果与一致
迭代 5: bs=4 结果与一致
迭代 6: bs=1 结果与一致
迭代 7: bs=2 结果与一致
迭代 8: bs=4 结果与一致
迭代 9: bs=4 结果与一致
迭代 10: bs=8 结果与一致



## 2 部分图编译

此示例展示如何将CUDA Graph应用于模型的一部分，在保持其余部分灵活性的同时，对频繁执行的子图进行加速。

模型ThreePartModel包含三个线性层（fc_a, fc_b, fc_c），每层后跟 ReLU（除了最后一层）。为简化，我们将fc_b单独捕获，而将ReLU放在图外处理（也可将ReLU包含在图内，但需确保其不会改变张量地址，通常in-place操作也可捕获，但为清晰起见，本例将激活函数放在图外）。

部分捕获：

- 捕获时只针对 model.fc_b(static_input_b)，不包含前后模块。

- 静态输入static_input_b的形状与模块A的输出一致，即[bs, 8, 64]

- 静态输出static_output_b的形状与模块B的输出一致，即[bs, 8, 128]


In [ ]:
import torch
import torch.nn as nn
import random

# 检查 CUDA 可用性
assert torch.cuda.is_available(), "CUDA 不可用"
device = torch.device('cuda')

# -------------------- 定义包含三个子模块的模型 --------------------
class ThreePartModel(nn.Module):
    def __init__(self):
        super().__init__()
        # 模块 A：线性 32 -> 64
        self.fc_a = nn.Linear(32, 64)
        # 模块 B：线性 64 -> 128
        self.fc_b = nn.Linear(64, 128)
        # 模块 C：线性 128 -> 32
        self.fc_c = nn.Linear(128, 32)
        self.relu = nn.ReLU()

    def forward(self, x):
        # x shape: [bs, seq_len=8, 32]
        x = self.relu(self.fc_a(x))           # [bs, 8, 64]
        x = self.relu(self.fc_b(x))           # [bs, 8, 128]
        x = self.fc_c(x)                      # [bs, 8, 32]
        return x

model = ThreePartModel().to(device)
model.eval()

# -------------------- 准备模块 B 的图池 --------------------
batch_sizes = [1, 2, 4, 8, 16]
graph_pool = {}  # bs -> (graph, static_input, static_output)

# 预热CUDA内核，避免捕获时包含自动调优过程
with torch.no_grad():
    dummy = torch.randn(8, 8, 32, device=device)
    for _ in range(3):
        _ = model(dummy)
torch.cuda.synchronize()

# 为每个batch size单独捕获模块B的图
for bs in batch_sizes:
    # 准备模块B的静态输入和输出
    static_input_b = torch.randn(bs, 8, 64, device=device)   # 与模块A的输出形状一致
    static_output_b = torch.empty(bs, 8, 128, device=device) # 模块B的输出形状

    # 捕获图
    graph = torch.cuda.CUDAGraph()
    with torch.cuda.graph(graph):
        # 在此上下文中只执行模块B
        static_output_b = model.fc_b(static_input_b)   # 注意：fc_b后没有relu，我们单独处理
        # 这里static_input_b和static_output_b的地址被记录
    # 将图和相关张量保存到池中
    graph_pool[bs] = (graph, static_input_b, static_output_b)

print(f"已为模块 B (batch sizes {batch_sizes}) 捕获图完成。")

# -------------------- 模拟多次推理，仅模块B使用图重放 --------------------
num_iterations = 10
for i in range(num_iterations):
    # 随机选择一个batch size
    bs = random.choice(batch_sizes)
    graph, static_in_b, static_out_b = graph_pool[bs]

    # 生成新的随机输入数据 (整个模型的输入)
    new_input = torch.randn(bs, 8, 32, device=device)

    # ----- 正常执行模块 A -----
    with torch.no_grad():
        out_a = model.relu(model.fc_a(new_input))   # [bs, 8, 64]

    # ----- 使用图重放模块 B -----
    # 将 out_a 复制到模块 B 的静态输入张量中（in-place 保持地址不变）
    static_in_b.copy_(out_a)
    # 重放图，此时 static_out_b 得到模块 B 的输出（注意图中没有 relu，需手动添加）
    graph.replay()
    # 对模块 B 的输出应用激活函数（因为图中未包含）
    out_b = model.relu(static_out_b)                # [bs, 8, 128]

    # ----- 正常执行模块 C -----
    with torch.no_grad():
        final_out = model.fc_c(out_b)               # [bs, 8, 32]

    # 验证：与完整模型普通前向的结果比较
    with torch.no_grad():
        expected = model(new_input)

    if torch.allclose(final_out, expected, atol=1e-5):
        print(f"迭代 {i+1}: bs={bs} 部分图重放结果与完整模型一致")
    else:
        print(f"迭代 {i+1}: bs={bs} 结果不一致！")


已为模块 B (batch sizes [1, 2, 4, 8, 16]) 捕获图完成。
迭代 1: bs=2 部分图重放结果与完整模型一致
迭代 2: bs=16 部分图重放结果与完整模型一致
迭代 3: bs=8 部分图重放结果与完整模型一致
迭代 4: bs=1 部分图重放结果与完整模型一致
迭代 5: bs=8 部分图重放结果与完整模型一致
迭代 6: bs=2 部分图重放结果与完整模型一致
迭代 7: bs=1 部分图重放结果与完整模型一致
迭代 8: bs=16 部分图重放结果与完整模型一致
迭代 9: bs=4 部分图重放结果与完整模型一致
迭代 10: bs=2 部分图重放结果与完整模型一致

